# **Project 1. Resume Analyzer**

In [1]:
pip install azure-ai-projects azure-core azure-storage-blob

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.3/274.3 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 8.3 MB/s eta 0:00:00


In [2]:
pip install --upgrade openai azure-ai-projects azure-identity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.5 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.31.0
    Uninstalling openai-2.31.0:
      Successfully uninstalled openai-2.31.0


In [4]:
from openai import OpenAI
import os
from google.colab import userdata

# Retrieve the secret key from Google Colab
api_key = userdata.get("safe")

client = OpenAI(
    api_key=api_key,  # Picks up the secret key from Colab environment
    base_url="https://email-assistant-resource.openai.azure.com/openai/v1"
)

# Note: api_version is usually NOT required when using this path in 2026

In [13]:
# 1. Define the input data
# In a real scenario, you can load this from a .txt or .json file
resume_data = """
RICHARD SANCHEZ
Education: Master of Business Management (2029-2030)
Work: Marketing Manager at Borcelle Studio (2030-Present)
Skills: Project Management, PR, Leadership...
"""

# 2. Use the Responses API for structured extraction
# Use the structured input pattern for GPT-4.1
# Updated call with 2026-compliant type naming
try:
    response = client.responses.create(
        model="gpt-4o",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",  # Changed from 'text'
                        "text": "Extract the following details into a JSON object: Name, Education, Latest Job, and Top 3 Skills. Return ONLY valid JSON."
                    },
                    {
                        "type": "input_text",  # Changed from 'text'
                        "text": f"RESUME SOURCE:\n{resume_text}"
                    }
                ]
            }
        ]
    )

    print("--- EXTRACTED DATA ---")
    print(response.output[0].content)

except Exception as e:
    print(f"Extraction Error: {e}")

--- EXTRACTED DATA ---
[ResponseOutputText(annotations=[], text='```json\n{\n  "Name": "Richard Sanchez",\n  "Education": "Master of Business Management (2029-2030)",\n  "Latest Job": "Marketing Manager at Borcelle Studio (2030-Present)",\n  "Top 3 Skills": ["Project Management", "PR", "Leadership"]\n}\n```', type='output_text', logprobs=[])]


In [17]:
import json

try:
    # Step 1: Get the response content
    output_text_item = response.output[0].content[0]

    # Step 2: Extract text (handles both dict and object cases)
    if isinstance(output_text_item, dict):
        raw_text = output_text_item.get("text", "")
    else:
        raw_text = output_text_item.text

    # Step 3: Clean the text (remove extra spaces/newlines)
    raw_text = raw_text.strip()

    # Step 4: Sometimes model adds extra text → extract only JSON part
    start = raw_text.find("{")
    end = raw_text.rfind("}") + 1
    raw_json_string = raw_text[start:end]

    # Step 5: Convert string → JSON
    extracted_data = json.loads(raw_json_string)

    print("✅ SUCCESS: Data Parsed")
    print(json.dumps(extracted_data, indent=2))

except Exception as e:
    print("Parsing Error:", e)
    print("Raw Output was:")
    print(raw_text)

✅ SUCCESS: Data Parsed
{
  "Name": "Richard Sanchez",
  "Education": "Master of Business Management (2029-2030)",
  "Latest Job": "Marketing Manager at Borcelle Studio (2030-Present)",
  "Top 3 Skills": [
    "Project Management",
    "PR",
    "Leadership"
  ]
}


In [20]:
from azure.storage.blob import BlobServiceClient

# Provide your Azure Storage connection string here
AZURE_STORAGE_CONNECTION_STRING = "DefaultEndpointsProtocol=https;AccountName=amnst;AccountKey=Pu+hTmAp9usLa4Idp+AHVVg+d7vSYy/zolLJmIVtI7j0NI2aAIhby8DgFR990FktyaitpRs/y+aY+ASt2JBVSA==;EndpointSuffix=core.windows.net"

# Create a BlobServiceClient to interact with the storage account
blob_service_client = BlobServiceClient.from_connection_string(
    AZURE_STORAGE_CONNECTION_STRING
)

# Name of the container where the file will be stored
container_name = "resume-output"

# Get a reference to the container
container_client = blob_service_client.get_container_client(container_name)

# Check if the container already exists
# If it does not exist, create it
if not container_client.exists():
    container_client.create_container()
    print("Container created successfully")
else:
    print("Container already exists")

# Create a BlobClient to upload a file
blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob="result.txt"
)

# Upload content to the blob
blob_client.upload_blob("Test upload", overwrite=True)

print("File uploaded successfully")

Container created successfully
File uploaded successfully


In [ ]:
import datetime

file_name = f"resume_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

In [ ]:
blob_client = blob_service_client.get_blob_client(
    container="resume-output",
    blob=file_name
)

blob_client.upload_blob(response.output_text, overwrite=True)

{'etag': '"0x8DE9EEE75C584E3"',
 'last_modified': datetime.datetime(2026, 4, 20, 15, 6, 54, tzinfo=datetime.timezone.utc),
 'content_md5': bytearray(b'\x0fi\xe9\x05\xc20a\x88g\xc3\xe0\xa8H\xf1\xef['),
 'client_request_id': '9185d488-3cca-11f1-b7f1-0242ac1c000c',
 'request_id': 'c40629bc-901e-0021-1fd7-d0e2a5000000',
 'version': '2026-02-06',
 'version_id': None,
 'date': datetime.datetime(2026, 4, 20, 15, 6, 54, tzinfo=datetime.timezone.utc),
 'request_server_encrypted': True,
 'encryption_key_sha256': None,
 'encryption_scope': None,
 'structured_body': None}

In [ ]:
print(f"✅ Stored as: {file_name}")

✅ Stored as: resume_20260420_150639.txt


Project2. AI Research Paper Analyzer

In [21]:
from openai import OpenAI
import os
from google.colab import userdata

# Retrieve the secret key from Google Colab
api_key = userdata.get("safe")

client = OpenAI(
    api_key=api_key,  # Picks up the secret key from Colab environment
    base_url="https://email-assistant-resource.openai.azure.com/openai/v1"
)

# Note: api_version is usually NOT required when using this path in 2026

In [22]:
# 1. Define the input data
# In a real scenario, this could come from a PDF, text file, or API
paper_text = """
Title: Deep Learning for Natural Language Processing

This paper explores how deep learning models such as transformers
have improved NLP tasks including translation, summarization,
and sentiment analysis. The study compares traditional methods
with modern neural approaches and highlights performance gains.
"""

# 2. Use the Responses API for summarization
try:
    response = client.responses.create(
        model="gpt-4o",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": (
                            "Summarize the following research paper into a JSON format with these fields: "
                            "Title, Problem Statement, Methodology, Key Findings, and Conclusion. "
                            "Keep it concise and return only valid JSON."
                        )
                    },
                    {
                        "type": "input_text",
                        "text": f"RESEARCH PAPER:\n{paper_text}"
                    }
                ]
            }
        ]
    )

    print("--- SUMMARY OUTPUT ---")
    print(response.output[0].content)

except Exception as e:
    print(f"Summarization Error: {e}")

--- SUMMARY OUTPUT ---
[ResponseOutputText(annotations=[], text='```json\n{\n  "Title": "Deep Learning for Natural Language Processing",\n  "Problem Statement": "Traditional NLP methods face limitations in handling complex language tasks, requiring improved models for enhanced performance.",\n  "Methodology": "The paper analyzes the impact of deep learning models, particularly transformers, by comparing their performance in translation, summarization, and sentiment analysis against traditional techniques.",\n  "Key Findings": "Deep learning approaches significantly outperform traditional methods across various NLP tasks, showcasing advancements in accuracy and efficiency.",\n  "Conclusion": "Transformers and other modern neural approaches enable breakthroughs in NLP, establishing them as superior solutions for complex language processing challenges."\n}\n```', type='output_text', logprobs=[])]


In [23]:
import json

try:
    # Step 1: Get the model response content
    output_text_item = response.output[0].content[0]

    # Step 2: Extract text safely (handles both dict and object formats)
    if isinstance(output_text_item, dict):
        raw_text = output_text_item.get("text", "")
    else:
        raw_text = output_text_item.text

    # Step 3: Clean extra spaces or newline characters
    raw_text = raw_text.strip()

    # Step 4: Extract only the JSON part from the response
    start_index = raw_text.find("{")
    end_index = raw_text.rfind("}") + 1
    json_string = raw_text[start_index:end_index]

    # Step 5: Convert string into JSON object
    summary_data = json.loads(json_string)

    print("Data parsed successfully")
    print(json.dumps(summary_data, indent=2))

except Exception as e:
    print("Error while parsing response:", e)
    print("Raw response received:")
    print(raw_text)

Data parsed successfully
{
  "Title": "Deep Learning for Natural Language Processing",
  "Problem Statement": "Traditional NLP methods face limitations in handling complex language tasks, requiring improved models for enhanced performance.",
  "Methodology": "The paper analyzes the impact of deep learning models, particularly transformers, by comparing their performance in translation, summarization, and sentiment analysis against traditional techniques.",
  "Key Findings": "Deep learning approaches significantly outperform traditional methods across various NLP tasks, showcasing advancements in accuracy and efficiency.",
  "Conclusion": "Transformers and other modern neural approaches enable breakthroughs in NLP, establishing them as superior solutions for complex language processing challenges."
}


In [24]:
from azure.storage.blob import BlobServiceClient
import json

# Provide your Azure Storage connection string
AZURE_STORAGE_CONNECTION_STRING = "DefaultEndpointsProtocol=https;AccountName=amnst;AccountKey=Pu+hTmAp9usLa4Idp+AHVVg+d7vSYy/zolLJmIVtI7j0NI2aAIhby8DgFR990FktyaitpRs/y+aY+ASt2JBVSA==;EndpointSuffix=core.windows.net"

# Create a BlobServiceClient
blob_service_client = BlobServiceClient.from_connection_string(
    AZURE_STORAGE_CONNECTION_STRING
)

# Container name for storing summaries
container_name = "research-summary-output"

# Get container client
container_client = blob_service_client.get_container_client(container_name)

# Create container if it does not exist
if not container_client.exists():
    container_client.create_container()
    print("Container created successfully")
else:
    print("Container already exists")

# Assume 'summary_data' contains parsed JSON from your summarizer
# Convert JSON object to string before uploading
summary_text = json.dumps(summary_data, indent=2)

# Create a blob client (file name can be dynamic)
blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob="paper_summary.json"
)

# Upload the summary to Azure Blob Storage
blob_client.upload_blob(summary_text, overwrite=True)

print("Research paper summary uploaded successfully")

Container created successfully
Research paper summary uploaded successfully


In [25]:
import datetime

file_name = f"paper_summary_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

In [26]:
# Create blob client for storing research paper summary
blob_client = blob_service_client.get_blob_client(
    container="research-summary-output",
    blob=file_name
)

# Convert summary JSON to string before uploading
summary_text = json.dumps(summary_data, indent=2)

# Upload the summary
blob_client.upload_blob(summary_text, overwrite=True)

print("Research paper summary uploaded successfully")

Research paper summary uploaded successfully


In [27]:
print(f"Stored research paper summary as: {file_name}")

Stored research paper summary as: paper_summary_20260421_070058.json


# **Project 3. AI Medical Report Interpreter**

In [28]:
from openai import OpenAI
import os
from google.colab import userdata

# Retrieve the secret key from Google Colab
api_key = userdata.get("safe")

client = OpenAI(
    api_key=api_key,  # Picks up the secret key from Colab environment
    base_url="https://email-assistant-resource.openai.azure.com/openai/v1"
)

# Note: api_version is usually NOT required when using this path in 2026

In [29]:
# 1. Define the input data
# In a real scenario, this could come from a PDF, lab report, or API
medical_report = """
Patient Name: John Doe
Age: 45

Symptoms:
- Fever
- Cough
- Fatigue

Test Results:
- Blood Pressure: 140/90 mmHg
- Blood Sugar: 180 mg/dL
- COVID-19 Test: Negative

Doctor Notes:
Patient shows signs of mild hypertension and elevated blood sugar.
Lifestyle changes and medication recommended.
"""

# 2. Use the Responses API for structured interpretation
try:
    response = client.responses.create(
        model="gpt-4o",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": (
                            "Analyze the following medical report and convert it into JSON format with these fields: "
                            "Patient Name, Age, Symptoms, Test Results, Diagnosis, Recommendations. "
                            "Keep it concise and return only valid JSON."
                        )
                    },
                    {
                        "type": "input_text",
                        "text": f"MEDICAL REPORT:\n{medical_report}"
                    }
                ]
            }
        ]
    )

    print("--- INTERPRETED MEDICAL REPORT ---")
    print(response.output[0].content)

except Exception as e:
    print(f"Medical Interpretation Error: {e}")

--- INTERPRETED MEDICAL REPORT ---
[ResponseOutputText(annotations=[], text='```json\n{\n  "Patient Name": "John Doe",\n  "Age": 45,\n  "Symptoms": ["Fever", "Cough", "Fatigue"],\n  "Test Results": {\n    "Blood Pressure": "140/90 mmHg",\n    "Blood Sugar": "180 mg/dL",\n    "COVID-19 Test": "Negative"\n  },\n  "Diagnosis": ["Mild Hypertension", "Elevated Blood Sugar"],\n  "Recommendations": ["Lifestyle changes", "Medication"]\n}\n```', type='output_text', logprobs=[])]


In [30]:
import json

try:
    # Step 1: Get the model response content
    output_text_item = response.output[0].content[0]

    # Step 2: Extract text safely (handles both dict and object formats)
    if isinstance(output_text_item, dict):
        raw_text = output_text_item.get("text", "")
    else:
        raw_text = output_text_item.text

    # Step 3: Clean extra spaces or newline characters
    raw_text = raw_text.strip()

    # Step 4: Extract only the JSON part from the response
    start_index = raw_text.find("{")
    end_index = raw_text.rfind("}") + 1
    json_string = raw_text[start_index:end_index]

    # Step 5: Convert string into JSON object
    medical_data = json.loads(json_string)

    print("✅ Medical report parsed successfully")
    print(json.dumps(medical_data, indent=2))

except Exception as e:
    print("❌ Error while parsing medical report:", e)
    print("Raw response received:")
    print(raw_text)

✅ Medical report parsed successfully
{
  "Patient Name": "John Doe",
  "Age": 45,
  "Symptoms": [
    "Fever",
    "Cough",
    "Fatigue"
  ],
  "Test Results": {
    "Blood Pressure": "140/90 mmHg",
    "Blood Sugar": "180 mg/dL",
    "COVID-19 Test": "Negative"
  },
  "Diagnosis": [
    "Mild Hypertension",
    "Elevated Blood Sugar"
  ],
  "Recommendations": [
    "Lifestyle changes",
    "Medication"
  ]
}


In [31]:
from azure.storage.blob import BlobServiceClient
import json

# Provide your Azure Storage connection string
AZURE_STORAGE_CONNECTION_STRING = "DefaultEndpointsProtocol=https;AccountName=amnst;AccountKey=Pu+hTmAp9usLa4Idp+AHVVg+d7vSYy/zolLJmIVtI7j0NI2aAIhby8DgFR990FktyaitpRs/y+aY+ASt2JBVSA==;EndpointSuffix=core.windows.net"

# Step 1: Create BlobServiceClient
blob_service_client = BlobServiceClient.from_connection_string(
    AZURE_STORAGE_CONNECTION_STRING
)

# Step 2: Container name for storing medical reports
container_name = "medical-report-output"

# Step 3: Get container client
container_client = blob_service_client.get_container_client(container_name)

# Step 4: Create container if it does not exist
if not container_client.exists():
    container_client.create_container()
    print("✅ Container created successfully")
else:
    print("📦 Container already exists")

# Step 5: Assume 'medical_data' contains parsed JSON from your AI model
# Convert JSON object to string
medical_text = json.dumps(medical_data, indent=2)

# Step 6: Create blob client (dynamic name recommended)
blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob="medical_report_summary.json"
)

# Step 7: Upload the medical report JSON
blob_client.upload_blob(medical_text, overwrite=True)

print("🏥 Medical report summary uploaded successfully")

✅ Container created successfully
🏥 Medical report summary uploaded successfully


In [32]:
import datetime

# Generate unique file name using current timestamp
file_name = f"medical_report_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

In [33]:
# Create blob client for storing medical report summary
blob_client = blob_service_client.get_blob_client(
    container="medical-report-output",
    blob=file_name
)

# Convert medical JSON to string before uploading
medical_text = json.dumps(medical_data, indent=2)

# Upload the medical report
blob_client.upload_blob(medical_text, overwrite=True)

print("🏥 Medical report summary uploaded successfully")

🏥 Medical report summary uploaded successfully


In [34]:
print(f"Stored medical report summary as: {file_name}")

Stored medical report summary as: medical_report_20260421_070849.json
